# NB2: Validating the Performance of Regional Climate Model Simulations (before and after Bias Correction)

## 1. Introduction
Regional Climate Models (RCMs) are one of the main tools used to investigate the impacts of climate change at regional and local scales. Unlike global climate models (GCMs), RCMs provide much higher spatial resolution, allowing a better representation of topography, coastlines and regional atmospheric processes.

Despite these advantages, RCM simulations usually exhibit systematic biases when compared with observational or reanalysis datasets. These biases arise from several sources, including imperfections in the driving GCM, limitations of the RCM itself and unresolved physical processes.

Because many climate impact studies require realistic representations of temperature and precipitation, it is common practice to apply bias correction methods before using RCM simluations.

This notebook introduces a complete validation workflow for CORDEX-CORE RCM simulations, using ERA5 as reference. Several diagnostics are computed before and after bias correction in order to assess the performance of different correction methods.

### 1.1. Learning Objectives
After completing this notebook, you should be able to

* understand the purpose of climate model validation;
* compare gridded climate datasets;
* compute validation metrics using the climate4R ecosystem;
* interpret model bias and temporal correlation;
* understand the rationale behind bias correction;
* compare different bias correction methods;
* evaluate climate indices before and after bias correction

## 2. Preparing the Working Environment
First, it is necessary to define a few parameters controlling the analysis to be undertaken and load the required libraries.

The climate4R ecosystem provides a collection of packages specifically designed for handling climate data. These packages  (`loadeR`, `transformeR`, `visualizeR`, `downscaleR`, `climate4R.value`) allow users to manipulate both station observations and gridded datasets through a common data structure, making subsequent analyses considerably easier (including temporal and spatial subsetting, regridding, visualization, bias correction and statistical downscaling).

In addition, some extra libraries outside climate4R (`RColorBrewer`, `lattice`, `gridExtra`) are also loaded to facilitate the production of publication-quality figures.

In [ ]:
rm(list = ls())

## params ##
dirbase = "/home/jovyan"
dirdata.obs = sprintf("%s/data/obs", dirbase)
dirdata.era5 = "https://thredds.climate.ifca.es/thredds/dodsC/fao/trainings/pakistan-202608/data/era5"
dirdata.cordex = "https://thredds.climate.ifca.es/thredds/dodsC/fao/trainings/pakistan-202608/data/cordex-core"

#lon = c(66, 75)
#lat = c(24, 35)

lon = c(66, 68)  # small region of study (encompassing Karachi)
lat = c(24, 26)  # small region of study (encompassing Karachi)

## libraries ##
library(loadeR)
library(transformeR)
library(visualizeR)
library(downscaleR)
library(climate4R.value)
library(RColorBrewer)
library(lattice)
library(gridExtra)

## auxiliary functions ##
source(sprintf("%s/notebooks/auxiliary_functions.R", dirbase))

## 3. Loading ERA5 and CORDEX-CORE Data

The first step consists of loading the two datasets that will be compared throughout the notebook (function `loadGridData()`).

On the one hand, the reference dataset is ERA5, a state-of-the-art atmospheric reanalysis that combines observations and numerical weather prediction through data assimilation.

On the other hand, we will consider simulations produced within the CORDEX-CORE initiative, one of the international coordinated efforts aimed at providing high-resolution regional climate projections. In particular, and for illustrative purposes, we will focus on the REMO2015 RCM, driven by the HadGEM2-ES GCM. Nonetheless, the full workflow described in this notebook could be equally applied to the rest of CORDEX-CORE simulations (e.g. GCM-RCM combinations) available.

Although both datasets describe the same atmospheric variable (daily maximum temperature), they originate from fundamentally different sources. Consequently, some discrepancies are expected and constitute the basis for the validation exercise presented in this notebook.

To alleviate a bit the computational cost of the workflow presented hereafter, the comparison between ERA5 and the CORDEX-CORE simulation (historical scenario) is carried out for the period 1986-2005.

In [ ]:
## loading tmax from ERA5 ##
era5 = loadGridData(sprintf("%s/tmax.nc", dirdata.era5),
                     var = "tmax", 
                     lonLim = ***, latLim = ***,
                     years = 1986:2005)
getShape(era5)

In [ ]:
## loading tmax from CORDEX-CORE ##
# model = c("HadGEM2-ES_REMO2015", "MPI-ESM-LR_REMO2015", "MPI-ESM-MR_RegCM4-7", "NorESM1-M_REMO2015","NorESM1-M_RegCM4-7")
model = c("HadGEM2-ES_REMO2015")
cordex = loadGridData(sprintf("%s/%s_historical_tmax.nc", dirdata.cordex, model),
                     var = "tmax", 
                     lonLim = ***, latLim = ***,
                     years = 1986:2005)
getShape(cordex)

Noteworthy, all the CORDEX-CORE simulations have been already regridded to the ERA5's regular grid of 0.25º resolution. 

In [ ]:
getGrid(era5)
identical(getGrid(era5), getGrid(cordex))

# 4. Harmonizing ERA5 and CORDEX-CORE Data

Even if both datasets cover approximately the same years, small differences in temporal coverage may exist because of missing records, different calendars or distinct processing procedures.

For this reason, the first preprocessing step consists of identifying the common temporal period shared by ERA5 and the CORDEX-CORE simulation (function `intersectGrid()`).

In [ ]:
era5 = intersectGrid(era5, ***, type = ***, which.return = 1)
getShape(era5)
getShape(cordex)

Both ERA5 and CORDEX-CORE store temperature in Kelvin (K). For interpretation and visualization purposes, both datasets are therefore converted from Kelvin to degrees Celsius before proceeding with the analysis.

In [ ]:
## converting K to degC ##
era5 = gridArithmetics(***)
cordex = gridArithmetics(***)

## 5. Validating CORDEX-CORE Data

No climate model is perfect. Therefore, before using it for impact studies, it is necessary to evaluate how realistically it reproduces present-day climate.

Model evaluation serves several purposes:

* identifying systematic biases;
* assessing temporal variability;
* evaluating spatial patterns;
* determining whether the simulations are suitable for a particular application.

The objective of validation is therefore not to determine whether a model is "correct", but rather to understand how it differs from the reference dataset and whether these differences are acceptable for the intended application.

The `climate4R.value` package implements a wide range of validation measures (see the `show.measures()` function).

Different metrics quantify different aspects of model performance.

For example,

* some evaluate temporal agreement;
* others assess spatial similarity;
* some measure systematic bias;
* others compare complete probability distributions.

Throughout this notebook we will progressively introduce several complementary diagnostics.

**Note:** Model evaluation should never rely on a single statistic.

### 5.1 Mean Bias
The simplest diagnostics used in climate model evaluation is the mean bias.

The bias is defined as the difference between the model climatology and the reference climatology. Positive values indicate systematic overestimation. Negative values indicate systematic underestimation.

Spatial bias maps are particularly useful because they reveal whether systematic errors are concentrated in specific geographical regions or whether they affect the entire domain.

In [ ]:
## computing (and plotting) the mean bias between ERA5 and CORDEX-CORE ##
bias = gridArithmetics(climatology(***), climatology(***), 
                       operator = ***)
getShape(bias)

In [ ]:
spatialPlot(***, 
            backdrop.theme = "countries",
            main = "ERA5 vs. CORDEX-CORE (tmax): Mean bias (ºC)",
            color.theme = "RdBu", rev.colors = TRUE,
            set.min = -5, set.max = 5, at = seq(-5, 5, 0.5))

### 5.2 Daily Correlation

The second validation metric considered here is the daily Pearson correlation coefficient (function `valueMeasure()` from package `climate4r.value`).

Correlation measures how similarly two time series evolve through time. A high correlation indicates that warm and cold periods tend to occur simultaneously in both datasets. Importantly, correlation does not account for model biases.

For example, a model may consistently underestimate temperature by several degrees while still exhibiting an excellent temporal correlation if it correctly reproduces the temporal variability.

In [ ]:
show.measures()  # metrics available in the climate4R.value package

## computing (and plotting) daily correlation between ERA5 and CORDEX-CORE ##
rho = valueMeasure(***, ***, measure.code = "ts.rp")
getShape(rho$***)

In [ ]:
spatialPlot(rho$***, 
           backdrop.theme = "countries",
           main = "ERA5 vs. CORDEX-CORE (tmax): Daily correlation",
           color.theme = "BuPu",
           set.min = 0, set.max = 1, at = seq(0, 1, 0.05))

### 5.3 Interannual Correlation

Daily variability is strongly influenced by individual weather systems.

Many climate impact studies, however, focus instead on variations from one year to another.

To investigate this aspect, both datasets are first aggregated into annual means. The correlation is then recomputed using these annual time series.

Because annual averaging filters out much of the short-term variability, interannual correlations often differ substantially from daily correlations. Comparing both diagnostics provides insight into the temporal scales that the model reproduces most successfully.

In [ ]:
## computing (and plotting) interannual correlation between ERA5 and CORDEX-CORE ##
rho.y = valueMeasure(***, ***, measure.code = "ts.rpY")
getShape(rho.y$***)

In [ ]:
spatialPlot(rho.y$***, 
            backdrop.theme = "countries",
            main = "ERA5 vs. CORDEX-CORE (tmax): Interannual correlation",
            color.theme = "BuPu",
            set.min = 0, set.max = 1, at = seq(0, 1, 0.05))

### 5.4 Bias in Extreme Percentiles (P95)
Beyond the mean bias, it is particularly interesting to investigate the bias in extreme temperatures, for instance, the 95th percentile.

In [ ]:
## computing (and plotting) the bias in P95 between ERA5 and CORDEX-CORE ##
p95.era5 = climatology(***)
p95.cordex = climatology(***)
bias.p95 = gridArithmetics(***)
getShape(bias.p95)

In [ ]:
spatialPlot(***, 
            backdrop.theme = "countries",
            main = "ERA5 vs. CORDEX-CORE (tmax): Bias for P95 (ºC)",
            color.theme = "RdBu", rev.colors = TRUE,
            set.min = -10, set.max = 10, at = seq(-10, 10, 1))

## 6. Bias Correction

Once systematic errors have been identified, the next logical question is whether they can be reduced. This is precisely the objective of bias correction.

Bias correction consists of statistically adjusting model simulations so that their statistical properties resemble those of a reference dataset during a historical calibration period. Typically, these adjustments are subsequently applied to future climate projections under the assumption that model biases remain approximately stationary over time.

Bias correction has become a standard preprocessing step in many climate change impact studies, particularly in agriculture, hydrology, ecology and energy applications.

However, it is important to remember that bias correction does not improve the physical realism of the climate model itself. Instead, it modifies the model output so that it becomes more suitable for downstream applications.

### 6.1 The Scaling Method

In climate4R, a number of bias correction methods (the most widely used in the literature) can be easily applied calling the `biasCorrection()` function from the `downscaleR` package. The simplest one is the Scaling method. 

For temperature variables, Scaling consists of adjusting the mean value (and only the mean) of the simulated series so that it matches the observed climatology.

Because only the average is modified, the method is computationally inexpensive and easy to interpret. However, its simplicity also limits its ability to correct other aspects of the distribution, such as variability or extremes.

**Note:** When the aim is to correct the historical scenario of a climate model, a cross-validation framework must be applied to avoid overfitting (`cross-val` argument). In this notebook, we use a k-fold approach, with k=2. 

In [ ]:
## application of the scaling method for bias correction ##
cordex.bc.scaling = biasCorrection(era5, cordex, 
                                   method = ***, 
                                   cross.val = ***, 
                                   folds = ***)

In [ ]:
## manually fixing bug with dates after bias correction ##
cordex.bc.scaling$Dates$start = as.character(cordex.bc.scaling$Dates$start)  
cordex.bc.scaling$Dates$end = as.character(cordex.bc.scaling$Dates$end) 

In [ ]:
getShape(cordex.bc.scaling)

In [ ]:
## alligning dates after bias correction ##
cordex.bc.scaling = intersectGrid(era5, ***, type = "temporal", which.return = 2)  

After applying the bias correction, the first step is to assess whether it has achieved its intended objective.

To evaluate the correction, we repeat exactly the same diagnostics that were previously computed for the raw simulation. Note that  comparing the same diagnostics before and after correction allows for assessing the effectiveness of the bias correction method objectively.

**Note:** Since Scaling only adjusts the mean climatology, we should expect a substantial reduction of the systematic bias. However, other characteristics of the distribution (for example, the bias in extreme percentiles) are unlikely to change. Likewise, bias correction are not designed to alter the temporal structure of the climate model. Thus, they are not expected to improve temporal correlation with observations.

In [ ]:
## mean bias ##
bias = gridArithmetics(***)
bias.bc.scaling = gridArithmetics(***)
spatialPlot(makeMultiGrid(***, ***), 
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(2, 1),
            main = "CORDEX-CORE bias (ºC), wrt ERA5",
            names.attr = c("raw", "scaling"),
            color.theme = "RdBu", rev.colors = TRUE,
            set.min = -5, set.max = 5, at = seq(-5, 5, 0.5))

In [ ]:
## daily correlation ##
rho = valueMeasure(***, ***, measure.code = ***)
rho.bc.scaling = valueMeasure(***, ***, measure.code = ***)

spatialPlot(makeMultiGrid(rho$***, rho.bc.scaling$***), 
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(2, 1),
            main = "Daily correlation between ERA5 and CORDEX-CORE",
            names.attr = c("raw", "scaling"),
            color.theme = "BuPu",
            set.min = 0, set.max = 1, at = seq(0, 1, 0.05))

In [ ]:
## interannual correlation ##
rho.y = valueMeasure(***, ***, measure.code = ***)
rho.y.bc.scaling = valueMeasure(***, ***, measure.code = ***))

spatialPlot(makeMultiGrid(rho.y$***, rho.y.bc.scaling$***), 
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(2, 1),
            main = "Interannual correlation between ERA5 and CORDEX-CORE",
            names.attr = c("raw", "scaling"),
            color.theme = "BuPu",
            set.min = 0, set.max = 1, at = seq(0, 1, 0.05))

In [ ]:
#################
## bias in P95 ##
#################

## computing (and plotting) P95 ##
p95.cordex.bc.scaling = climatology(***)

spatialPlot(makeMultiGrid(p95.era5, p95.cordex, p95.cordex.bc.scaling), 
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(3, 1),
            main = "P95 (ºC)",
            names.attr = c("ERA5", "CORDEX-CORE (raw)", "CORDEX-CORE (scaling)"),
            color.theme = "YlOrRd",
            set.min = 0, set.max = 50, at = seq(0, 50, 2.5))

In [ ]:
## computing (and plotting) biases for P95 ##
bias.p95.cordex = gridArithmetics(***)
bias.p95.cordex.bc.scaling = gridArithmetics(***)
spatialPlot(makeMultiGrid(***, ***),
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(2, 1),
            main = "Bias in P95 (ºC), wrt ERA5",
            names.attr = c("CORDEX-CORE (raw)", "CORDEX-CORE (scaling)"),
            color.theme = "RdBu", rev.colors = TRUE,
            set.min = -10, set.max = 10, at = seq(-10, 10, 1))

### 6.2 The Empirical Quantile Mapping (EQM) Method
The second bias correction technique explored in this notebook is Empirical Quantile Mapping (EQM).

Unlike Scaling, EQM does not simply adjust the mean value. Instead, it compares the empirical cumulative distribution functions of the reference dataset and the model simulation during the calibration period. For each simulated value, the method identifies its corresponding percentile in the model distribution and then replaces it with the value associated with the same percentile in the reference distribution. As a result, the corrected simulation reproduces not only the observed mean but also the observed variability and much of the distributional shape.

Because of its flexibility, EQM has become one of the most widely used bias adjustment methods in climate science.

In [ ]:
## application of the EQM method for bias correction ##
cordex.bc.eqm = biasCorrection(era5, cordex, 
                               method = ***, 
                               cross.val = ***, 
                               folds = ***)

In [ ]:
## manually fixing bug with dates after bias correction ##
cordex.bc.eqm$Dates$start = as.character(cordex.bc.eqm$Dates$start)  
cordex.bc.eqm$Dates$end = as.character(cordex.bc.eqm$Dates$end) 

getShape(cordex.bc.eqm)

In [ ]:
## alligning dates after bias correction ##
cordex.bc.eqm = intersectGrid(era5, ***, type = "temporal", which.return = 2) 

Once EQM has been applied, the same validation diagnostics considered to evaluate the performance of the Scaling method are computed again. Note that, maintaining an identical evaluation framework allows the relative performance of the two methods to be compared directly.

In [ ]:
## mean bias ##
bias = gridArithmetics(***)
bias.bc.scaling = gridArithmetics(***)
bias.bc.eqm = gridArithmetics(***)
spatialPlot(makeMultiGrid(***, ***, ***), 
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(3, 1),
            main = "CORDEX-CORE: Mean bias (ºC), wrt ERA5",
            names.attr = c("raw", "scaling", "eqm"),
            color.theme = "RdBu", rev.colors = TRUE,
            set.min = -5, set.max = 5, at = seq(-5, 5, 0.5))

In [ ]:
## daily correlation ##
rho = valueMeasure(***)
rho.bc.scaling = valueMeasure(***)
rho.bc.eqm = valueMeasure(***)

spatialPlot(makeMultiGrid(***, ***, ***), 
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(3, 1),
            main = "Daily correlation between ERA5 and CORDEX-CORE",
            names.attr = c("raw", "scaling", "eqm"),
            color.theme = "BuPu",
            set.min = 0, set.max = 1, at = seq(0, 1, 0.05))

In [ ]:
## interannual correlation ##
rho.y = valueMeasure(***)
rho.y.bc.scaling = valueMeasure(***)
rho.y.bc.eqm = valueMeasure(***)

spatialPlot(makeMultiGrid(***, ***, ***), 
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(3, 1),
            main = "Interannual correlation between ERA5 and CORDEX-CORE",
            names.attr = c("raw", "scaling", "eqm"),
            color.theme = "BuPu",
            set.min = 0, set.max = 1, at = seq(0, 1, 0.05))

In [ ]:
## bias in P95 ##
p95.cordex = climatology(***)
bias.p95.cordex = gridArithmetics(***)

p95.cordex.bc.scaling = climatology(***)
bias.p95.cordex.bc.scaling = gridArithmetics(***)

p95.cordex.bc.eqm = climatology(***)
bias.p95.cordex.bc.eqm = gridArithmetics(***)

spatialPlot(makeMultiGrid(***, ***, ***),
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(3, 1),
            main = "CORDEX-CORE: Bias in P95 (ºC), wrt ERA5",
            names.attr = c("CORDEX-CORE (raw)", "CORDEX-CORE (scaling)", "CORDEX-CORE (eqm)"),
            color.theme = "RdBu", rev.colors = TRUE,
            set.min = -10, set.max = 10, at = seq(-10, 10, 1))

## 7. Climate Indices

Many climate impact studies do not analyse climatic variables (e.g. temperature, precipitation, etc.) directly.

Instead, they focus on climate indices, i.e. variables derived from the original meteorological observations that describe phenomena of practical interest.

Some examples are:

* the number of frost days;
* the number of tropical nights;
* the number and duration of heat waves;
* the number of rainy indices;
* etc.
  
Climate indices often provide a much more direct connection between climate and societal impacts than raw meteorological variables.

In this notebook, we investigate a commonly used temperature-derived indicator: the number of days per year with maximum temperature above 35ºC (ND35). The function `binaryGrid()` is used to compute it.

Note that, threshold-based indices are widely used because they are easy to interpret and often correspond directly to impact-relevant conditions.

For example,

* agricultural productivity
* livestock heat stress
* electricity demand
* human thermal comfort

may all depend on the number of days exceeding particular temperature thresholds.

**Note:** Unlike percentile-based indices, threshold indices are especially sensitive to systematic biases. Even a relatively small temperature bias can produce large differences in the number of exceedances. This explains why bias correction has become standard practice in sectors such as agriculture and human health.

In [ ]:
########################################################################################################
## computing (and comparing) ND35 for ERA5 and CORDEX-CORE (raw, scaling-corrected and EQM-corrected) ##
########################################################################################################

## computing ND35 ##
binaryGrid(era, condition = ***, threshold = ***)
aggregateGrid(***)
climatology(***)

In [ ]:
nd35.era5 = *** 
nd35.cordex = ***
nd35.cordex.bc.scaling = ***
nd35.cordex.bc.eqm = ***

In [ ]:
## plotting ##
spatialPlot(makeMultiGrid(nd35.era5, nd35.cordex, nd35.cordex.bc.scaling, nd35.cordex.bc.eqm), 
            backdrop.theme = "countries",
            main = "ND35",
            as.table = TRUE,
            layout = c(4, 1),
            names.attr = c("ERA5", "CORDEX-CORE (raw)", 
                           "CORDEX-CORE (scaling)", "CORDEX-CORE (eqm)"),
            color.them = "YlOrRd",
            set.min = 0, set.max = 300, at = seq(0, 300, 20))

## 8. From Spatial to Local Validation

So far, the evaluation has focused primarily on spatial diagnostics. Maps are very useful because they provide an overview of model performance across the entire study domain.

However, many climate impact studies are ultimately conducted at specific locations. Farmers, water managers and urban planners are usually interested in the climate conditions at individual sites. For this reason, we now shift our attention from the spatial domain to the analysis of a single location.

This change of perspective allows us to investigate aspects of model performance that are difficult to appreciate from maps alone, including the detailed temporal evolution and the complete statistical distribution of the simulated temperatures.

The station considered for this analysis is Karachi Airport, one of the best-known meteorological stations in Pakistan. The station is retrieved through its metadata rather than by its numerical position within the dataset.

In [ ]:
## Karachi (airport) ##
lon.karachi = 67.133 
lat.karachi = 24.9

era5.karachi = subsetGrid(era5, ***)
cordex.karachi = subsetGrid(cordex, ***)
cordex.bc.scaling.karachi = subsetGrid(cordex.bc.scaling, ***)
cordex.bc.eqm.karachi = subsetGrid(cordex.bc.eqm, ***)

First, we produce q-q plots for comparing the daily observed records against ERA5, the raw and the bias-corrected (both using the scaling and the EQM methods) CORDEX-CORE simulations at Karachi (airport).

In [ ]:
## q-q plots, for karachi ##
par(mfrow = c(1, 3))
qqplot(era5.karachi$***, cordex.karachi$***, 
              main = "Q-Q plot (tmax)",
              xlab = "ERA5", ylab = "CORDEX"); abline(0, 1, col = "red"); grid()
qqplot(era5.karachi$***, cordex.bc.scaling.karachi$***, 
              main = "Q-Q plot (tmax)",
              xlab = "ERA5", ylab = "CORDEX (scaling)"); abline(0, 1, col = "red"); grid()
qqplot(era5.karachi$***, cordex.bc.eqm.karachi$***, 
              main = "Q-Q plot (tmax)",
              xlab = "ERA5", ylab = "CORDEX (eqm)"); abline(0, 1, col = "red"); grid()
dev.off()

Second, we compare the PDF derived from daily observed records against those corresponding to ERA5, the raw and the bias-corrected (both using the scaling and the EQM methods) CORDEX-CORE simulations at Karachi (airport).

In [ ]:
## PDFs
pdf.era5.karachi = density(***)
pdf.cordex.karachi = density(***)
pdf.cordex.bc.scaling.karachi = density(***, na.rm = TRUE)
pdf.cordex.bc.eqm.karachi = density(***, na.rm = TRUE)

In [ ]:
plot(pdf.era5.karachi, col = "black", lwd = 2,
     main = "PDFs: Karachi",
     xlab = "Tmax (ºC)", ylab = "Density",
     xlim = c(5, 55))

In [ ]:
lines(pdf.cordex.karachi, col = "blue", lwd = 2)

In [ ]:
lines(pdf.cordex.bc.scaling.karachi, col = "red", lwd = 2, lty = 2)

In [ ]:
lines(pdf.cordex.bc.eqm.karachi, col = "green", lwd = 2, lty = 2)

In [ ]:
legend("topleft", 
       legend = c("ERA5", "CORDEX-CORE (raw)", "CORDEX-CORE (scaling)", "CORDEX-CORE (eqm)"),
       col = c("black", "blue", "red", "green"),
       lwd = 2,
       cex = 0.5)
grid()

Finally, we compute and visualize the ND35 for ERA5, the raw and the bias-corrected (both using the scaling and the EQM methods) CORDEX-CORE simulations at Karachi (airport).

In [ ]:
## ND35, for karachi ##
nd35.era5.karachi = ***
nd35.cordex.karachi = ***
nd35.cordex.bc.scaling.karachi = ***
nd35.cordex.bc.eqm.karachi = ***

## plotting time series ##
temporalPlot(nd35.era5.karachi, nd35.cordex.karachi, 
            nd35.cordex.bc.scaling.karachi,
            nd35.cordex.bc.eqm.karachi,
            cols = c("black", "blue", "red", "green"))